# Credit Risk ML Case Study: Model Evaluation

This notebook evaluates probability-of-default models trained on the Give Me Some Credit dataset. The focus is practical model assessment for credit-risk and insurance-style analytics: ranking borrowers by risk, checking whether predicted probabilities are usable, and summarizing results in a business-facing way.

## Business Framing

The target variable is serious delinquency within two years. Model scores are interpreted as predicted probability of default (PD), not only as class labels.

For risk modelling, three perspectives matter at the same time:

- **Ranking:** higher-risk borrowers should receive higher PDs than lower-risk borrowers.
- **Calibration:** predicted PDs should be close to observed default rates if they are used in decision-making.
- **Segmentation:** risk bands or deciles should separate the portfolio into useful business groups.

## Setup

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import evaluate as evaluate_module
import plots as plots_module

importlib.reload(evaluate_module)
importlib.reload(plots_module)

from evaluate import (
    compare_model_metrics,
    make_calibration_table,
    make_decile_table,
    save_table,
)
from plots import (
    plot_calibration_curve,
    plot_decile_default_rate,
    plot_pd_distribution,
    plot_precision_recall_curve,
    plot_roc_curve,
    save_figure,
)

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

METRICS_PATH = TABLES_DIR / "model_metrics.csv"
PREDICTIONS_PATH = TABLES_DIR / "validation_predictions.csv"
COEFFICIENTS_PATH = TABLES_DIR / "logistic_coefficients.csv"

pd.set_option("display.float_format", "{:.4f}".format)

## Load Validation Predictions And Metrics

This notebook consumes the outputs from `02_model_training.ipynb`. It does not create placeholder metrics if the training outputs are missing.

In [ ]:
required_paths = {
    "model metrics": METRICS_PATH,
    "validation predictions": PREDICTIONS_PATH,
    "logistic coefficients": COEFFICIENTS_PATH,
}

missing_paths = [
    f"{label}: {path.relative_to(PROJECT_ROOT)}"
    for label, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    message = (
        "Training outputs are missing. Run notebooks/02_model_training.ipynb first.\n"
        + "\n".join(f"- {item}" for item in missing_paths)
    )
    raise FileNotFoundError(message)

training_metrics = pd.read_csv(METRICS_PATH)
validation_predictions = pd.read_csv(PREDICTIONS_PATH)
logistic_coefficients = pd.read_csv(COEFFICIENTS_PATH)

validation_predictions.head()

In [ ]:
LOGISTIC_PD_COL = "logistic_l2_clipped_pd"
XGBOOST_PD_COL = "xgboost_pd"

required_prediction_columns = {"y_true", LOGISTIC_PD_COL, XGBOOST_PD_COL}
missing_prediction_columns = required_prediction_columns - set(validation_predictions.columns)

if missing_prediction_columns:
    raise ValueError(
        "validation_predictions.csv is missing expected columns: "
        f"{sorted(missing_prediction_columns)}"
    )

y_true = validation_predictions["y_true"].astype(int)
model_scores = {
    "Logistic regression": validation_predictions[LOGISTIC_PD_COL],
    "XGBoost": validation_predictions[XGBOOST_PD_COL],
}

training_metrics

## Model Comparison Table

The table below recomputes the validation metrics from the saved prediction file. This keeps the evaluation notebook independent from any displayed output in the training notebook.

In [ ]:
comparison_table = compare_model_metrics(
    {
        model_name: (y_true, pd_scores)
        for model_name, pd_scores in model_scores.items()
    }
)

comparison_table.style.format(
    {
        "roc_auc": "{:.4f}",
        "pr_auc": "{:.4f}",
        "log_loss": "{:.4f}",
        "brier_score": "{:.4f}",
        "default_rate": "{:.2%}",
        "mean_predicted_pd": "{:.2%}",
    }
)

## ROC-AUC And PR-AUC Curves

ROC-AUC summarizes ranking across thresholds. Precision-recall AUC (PR-AUC) is especially useful here because serious delinquency is a minority event. The AUC values are shown directly in the plot legends.

In [ ]:
fig, ax = plot_roc_curve(y_true, model_scores)
save_figure(fig, FIGURES_DIR / "roc_auc_curve.png")
plt.show()

In [ ]:
fig, ax = plot_precision_recall_curve(y_true, model_scores)
save_figure(fig, FIGURES_DIR / "pr_auc_curve.png")
plt.show()

## Calibration Analysis

Good ranking is not enough for credit-risk decision-making. If a score is used as a probability of default, predicted PDs should be reasonably close to observed default rates within risk bands.

The calibration points below are bin averages, not individual maximum scores. Because most validation predictions are in a low-PD range, the calibration view uses logarithmic axes to make the low-risk region easier to inspect while keeping the perfect-calibration reference meaningful.

In [ ]:
calibration_tables = {
    model_name: make_calibration_table(y_true, pd_scores, n_bins=10)
    for model_name, pd_scores in model_scores.items()
}

for model_name, table in calibration_tables.items():
    save_table(
        table,
        TABLES_DIR / f"calibration_{model_name.lower().replace(' ', '_')}.csv",
    )

fig, ax = plot_calibration_curve(calibration_tables, log_scale=True)
save_figure(fig, FIGURES_DIR / "calibration_curve.png")
plt.show()

## Predicted PD Distribution

The predicted PD distribution shows how concentrated or dispersed the model scores are. For business users, this helps explain whether the model creates meaningful risk separation across the portfolio.

Most borrowers have low predicted PDs, so a log-scaled x-axis is also useful for inspecting the low-risk region without losing the high-risk tail.

In [ ]:
fig, ax = plot_pd_distribution(model_scores)
save_figure(fig, FIGURES_DIR / "pd_distribution.png")
plt.show()

In [ ]:
fig, ax = plot_pd_distribution(model_scores, log_x=True)
save_figure(fig, FIGURES_DIR / "pd_distribution_log_scale.png")
plt.show()

## Risk Decile Analysis

Risk deciles sort borrowers by predicted PD. The highest-risk deciles should show much higher observed default rates than the portfolio average. This view is useful for risk segmentation, underwriting review, monitoring, and business reporting.

In [ ]:
decile_tables = {
    model_name: make_decile_table(y_true, pd_scores, n_bins=10)
    for model_name, pd_scores in model_scores.items()
}

for model_name, table in decile_tables.items():
    save_table(
        table,
        TABLES_DIR / f"deciles_{model_name.lower().replace(' ', '_')}.csv",
    )

decile_tables["XGBoost"].style.format(
    {
        "mean_predicted_pd": "{:.2%}",
        "observed_default_rate": "{:.2%}",
        "min_predicted_pd": "{:.2%}",
        "max_predicted_pd": "{:.2%}",
        "lift_vs_portfolio_default_rate": "{:.2f}x",
    }
)

In [ ]:
for model_name, table in decile_tables.items():
    fig, ax = plot_decile_default_rate(table, model_name=model_name)
    save_figure(
        fig,
        FIGURES_DIR / f"decile_default_rate_{model_name.lower().replace(' ', '_')}.png",
    )
    plt.show()

## Logistic Regression Interpretability

The logistic regression baseline is useful because its coefficients can be inspected. Since the model is fitted after standardization, the odds ratios below represent the multiplicative change in odds for a one-standard-deviation increase in the feature, after preprocessing.

In [ ]:
logistic_coefficients.head(12).style.format(
    {
        "coefficient": "{:.4f}",
        "odds_ratio_per_std": "{:.3f}",
    }
)

## XGBoost Feature Importance

TODO: Add a simple XGBoost feature-importance view after the training workflow saves the fitted model or exports feature importances. This notebook currently uses saved validation predictions and does not reload a fitted booster. SHAP is intentionally skipped for now to keep the case study lightweight.

## Limitations

- The dataset is a public Kaggle dataset and is not a live lending or insurance portfolio.
- There is no external validation sample beyond the train-validation split.
- Fairness and protected-class analysis are not included yet.
- This is not a production deployment and does not include model serving infrastructure.
- Real-time monitoring is not implemented.
- Business-specific costs, pricing rules, and risk appetite thresholds are not included.

## Next Steps

- Create a short model card summarizing intended use, validation data, metrics, and limitations.
- Add basic monitoring checks for score drift, default-rate drift, missing-value rates, and calibration stability.
- Add lightweight explainability outputs for XGBoost, starting with feature importance before considering SHAP.
- Add AI-assisted business reporting in English and German, using the saved tables to generate concise stakeholder summaries without changing model results.